In [46]:
# Python 3.10.11
# %pip install -r requirements.txt > /dev/null
from args import *
from utils import *

In [47]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize

from torchmetrics import AUROC
from torch.utils.tensorboard import SummaryWriter


# 依赖导入
import pandas as pd


In [48]:
balance = True


# 导入label数据
label_df = pd.read_csv(sample_labels_file_path, index_col=0)

# 裁减样本数量，使得 0 - 1 样本数量一致
df_0 = label_df[label_df['label'] == 0]
df_1 = label_df[label_df['label'] == 1]
min_len = min(len(df_0), len(df_1))
new_df = pd.concat([df_0[:min_len], df_1[:min_len]], axis=0)

sample_key_list = label_df.index.to_list()
if balance:
    sample_key_list = new_df.index.to_list() # 均衡数据

print(f"label 样本数量: {len(sample_key_list)}")

# TODO(241225) 导入gene数据
# 根据 sample_key_list 为基准, 若模态数据中不存在 sample_key 则填充新数据
origin_gene_array = load_gene_data_by_sample_key(sample_key_list).values
origin_cnv_array = load_cnv_data_by_sample_key(sample_key_list).values

origin_wsi_array = load_wsi_data_by_sample_key(sample_key_list)
origin_report_array = load_report_data_by_sample_key(sample_key_list)

label 样本数量: 152


In [49]:
def get_mode(mode_name, column_len=1_000_000):

    global origin_gene_array, origin_cnv_array, origin_wsi_array, origin_report_array

    mode_list = []

    for key in mode_name:
        
        if key == "g":
            mode = np.copy(origin_gene_array)
        elif key == "c":
            mode = np.copy(origin_cnv_array)
        elif key == "w":
            mode = np.copy(origin_wsi_array)
        elif key == "r":
            mode = np.copy(origin_report_array)
        mode_list.append(mode[:, :column_len])
    return tuple(mode_list)


In [50]:
# TODO 模态类别
mode_name = "gcwr"

gene_array, cnv_array, wsi_array, report_array = get_mode(mode_name)

In [51]:
gene_array.shape, cnv_array.shape, wsi_array.shape, report_array.shape

((152, 2340), (152, 2340), (152, 2048), (152, 768))

In [52]:
label_array = label_df.values
if balance:
    label_array = new_df.values # 均衡数据

_, gene_dim = gene_array.shape
_, cnv_dim = cnv_array.shape
_, wsi_dim = wsi_array.shape
_, report_dim = report_array.shape
_, label_dim = label_array.shape

print(f"""
gene 数据维度:   {gene_dim}
cnv 数据维度:    {cnv_dim}
wsi 数据维度:    {wsi_dim}
report 数据维度: {report_dim}
""")

batch_size = 32

all_dataset = MultiOmicsDataset(gene_array, cnv_array, report_array, wsi_array, label_array)

# 假设 all_dataset 是一个 Dataset 对象
train_len = int(len(all_dataset) * 0.8)  # 80% 的数据用作训练集
test_len = len(all_dataset) - train_len  # 剩余的数据用作验证集

# 使用 random_split 分割数据集
train_val_dataset, test_dataset = random_split(all_dataset, [train_len, test_len])

train_loader = DataLoader(all_dataset, batch_size=batch_size, shuffle=True, num_workers=3, drop_last=False)
val_loader = DataLoader(train_val_dataset, batch_size=batch_size, shuffle=False, num_workers=3, drop_last=False)


gene 数据维度:   2340
cnv 数据维度:    2340
wsi 数据维度:    2048
report 数据维度: 768



In [53]:
### 分割线

In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# 构造含交叉注意力机制的Transformer
class TransformerEncoderLayerWithCrossAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1, dim_feedforward=2048,):
        super(TransformerEncoderLayerWithCrossAttention, self).__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)

        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, src, src_mask=None):

                # 自注意力机制
        src2 = self.self_attn(src, src, src, attn_mask=src_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)

        # 将输入序列平均拆分为4等分
        seq_len = src.size(1)
        seg_len = seq_len // 4
        seg1 = src[:, :seg_len, :]
        seg2 = src[:, seg_len:2*seg_len, :]
        seg3 = src[:, 2*seg_len:3*seg_len, :]
        seg4 = src[:, 3*seg_len:, :]

        # 交叉注意力机制
        seg1_cross, _ = self.cross_attn(seg1, seg2, seg2)
        seg2_cross, _ = self.cross_attn(seg2, seg3, seg3)
        seg3_cross, _ = self.cross_attn(seg3, seg4, seg4)
        seg4_cross, _ = self.cross_attn(seg4, seg1, seg1)

        # 合并交叉注意力结果
        src_cross = torch.cat([seg1_cross, seg2_cross, seg3_cross, seg4_cross], dim=1)
        src = src + self.dropout2(src_cross)
        src = self.norm2(src)

        # 线性层和残差连接
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        src = src + self.dropout3(src2)
        src = self.norm3(src)

        return src

In [55]:
class TransformerEncoderWithCrossAttention(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None):
        super(TransformerEncoderWithCrossAttention, self).__init__()
        self.layers = nn.ModuleList([encoder_layer for _ in range(num_layers)])
        self.num_layers = num_layers
        self.norm = norm

    def forward(self, src, mask=None):
        output = src

        for layer in self.layers:
            output = layer(output, src_mask=mask)

        if self.norm is not None:
            output = self.norm(output)

        return output

# 定义模型结构
class MultiOmicsModel(nn.Module):
    def __init__(self, dropout_prob=0.2):
        super(MultiOmicsModel, self).__init__()

        # 分割线
        global gene_dim, cnv_dim, report_dim, wsi_dim, label_dim

        same_all_feature_dim = 128
        self.shared_hidden_gene_cnv = nn.Linear(gene_dim + cnv_dim, gene_dim)
        self.hidden_gene = nn.Sequential(nn.Linear(gene_dim, same_all_feature_dim))
        self.hidden_cnv = nn.Sequential(nn.Linear(gene_dim, same_all_feature_dim))

        self.fc_report = nn.Linear(report_dim, same_all_feature_dim)
        self.fc_wsi = nn.Linear(wsi_dim, same_all_feature_dim)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_prob)

        

        # 创建一个带有交叉注意力的Transformer编码器层
        # nn.MultiheadAttention(d_model, nhead, dropout=dropout) 
        d_model, nhead, dropout = same_all_feature_dim, 1, 0.1
        dim_feedforward = 32
        num_layers = 1
        encoder_layer = TransformerEncoderLayerWithCrossAttention(d_model, nhead, dropout, dim_feedforward)
        # 创建一个带有交叉注意力的Transformer编码器
        self.encoder = TransformerEncoderWithCrossAttention(encoder_layer, num_layers)

        self.flatten = nn.Flatten(start_dim=1)

        # 输出层
        self.lin = nn.Linear(same_all_feature_dim * 4, 2)


    def forward(self, gene_tensor, cnv_tensor, report_tensor, wsi_tensor):

        gene_cnv_feature = torch.concat([gene_tensor, cnv_tensor], dim=1)
        gene_cnv_feature = self.relu(self.shared_hidden_gene_cnv(gene_cnv_feature))
        gene_cnv_feature = self.dropout(gene_cnv_feature)

        report_feature = self.fc_report(report_tensor)
        report_feature = self.relu(report_feature)

        wsi_feature = self.fc_wsi(wsi_tensor)
        wsi_feature = self.relu(wsi_feature)

        gene_feature = self.hidden_gene(gene_cnv_feature)
        gene_feature = self.relu(gene_feature)

        cnv_feature = self.hidden_cnv(gene_cnv_feature)
        cnv_feature = self.relu(cnv_feature)

        all_feature = torch.stack([gene_feature, cnv_feature, report_feature, wsi_feature])
        all_feature = self.dropout(all_feature)

        # all_feature = all_feature.permute(1, 0, 2)
        # all_feature = self.flatten(all_feature)
        
        # out = self.lin(all_feature)
        # return out

        """
        start:  torch.Size([4, 32, 64])
        before enccode:  torch.Size([32, 4, 64])
        after enccode:  torch.Size([32, 4, 64])
        end:  torch.Size([4, 32, 64])
        """
        all_feature = all_feature.permute(1,0,2)
        all_feature = self.encoder(all_feature)

        all_feature = all_feature.permute(1, 0, 2)
        all_feature = self.relu(all_feature)

        all_feature = self.dropout(all_feature)

        all_feature = all_feature.permute(1, 0, 2)
        all_feature = self.flatten(all_feature)
        
        out = self.lin(all_feature)
        return out

In [56]:
print(f"""
gene 数据维度:   {gene_dim}
cnv 数据维度:    {cnv_dim}
wsi 数据维度:    {wsi_dim}
report 数据维度: {report_dim}
""")

# model(expr, cnv, report, sis)

# learning_rate = 0.001
# criterion = nn.BCEWithLogitsLoss()
# optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()  # 适用于二分类问题

# 
model = MultiOmicsModel(0.290005700).to(device) # 250120-
optimizer = optim.Adam(model.parameters(), lr=0.00011561635015) # 250120-


gene 数据维度:   2340
cnv 数据维度:    2340
wsi 数据维度:    2048
report 数据维度: 768



In [57]:
from thop import profile

# 假设 model 是你的 PyTorch 模型实例
# input_tensor 是一个代表输入数据的张量，其形状应与模型的输入层匹配
gene_tensor, cnv_tensor = torch.randn(1, gene_array.shape[1]), torch.randn(1, cnv_array.shape[1])
report_tensor, wsi_tensor = torch.rand(1, report_array.shape[1]), torch.randn(1, wsi_array.shape[1])

gene_tensor = gene_tensor.to(device)
cnv_tensor = cnv_tensor.to(device)
report_tensor = report_tensor.to(device)
wsi_tensor = wsi_tensor.to(device)

# 计算 FLOPs
flops, params = profile(model, inputs=(gene_tensor,cnv_tensor,report_tensor,wsi_tensor,))
print(f'FLOPs: {flops}')
print(f'Parameters: {params}')

[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
FLOPs: 11950624.0
Parameters: 11923686.0


In [58]:
from sklearn.metrics import roc_auc_score

def show_score(score, can_print=False, tb_index=None, fold_idx=0, epoch_idx=0):
    auc = score["auc"]
    accuracy = score["accuracy"]
    f1_score = score["f1_score"]
    recall = score["recall"]
    specificity = score["specificity"]
    precision = score["precision"]
    mcc = score["mcc"]
    if can_print:
        # print("序号 Auc Acc F1-score Pre Sn Sp Mcc")
        print(f"{fold_idx + 1}-{epoch_idx + 1}", 
              f"{auc:.4f}", 
              f"{accuracy:.4f}", 
              f"{f1_score:.4f}", 
              f"{precision:.4f}", 
              f"{recall:.4f}", 
              f"{specificity:.4f}", 
              f"{mcc:.4f}")        
        return torch.tensor([auc, accuracy, f1_score, precision, recall, specificity, mcc])
    # if can_print:
    #     print("AUC:", auc)
    #     print("Accuracy:", accuracy)
    #     print("F1 Score:", score["f1_score"])
    #     print("Recall:", f1_score)
    #     print("Specificity:", recall)
    #     print("Precision:", precision)
    #     print("MCC:", mcc)
    if tb_index is None:
        return
    # global writer
    # # 将指标写入 TensorBoard
    # writer.add_scalar('Metrics/AUC', score["auc"], tb_index)
    # writer.add_scalar('Metrics/Accuracy', score["accuracy"], tb_index)
    # writer.add_scalar('Metrics/F1_Score', score["f1_score"], tb_index)
    # writer.add_scalar('Metrics/Recall', score["recall"], tb_index)
    # writer.add_scalar('Metrics/Specificity', score["specificity"], tb_index)
    # writer.add_scalar('Metrics/Precision', score["precision"], tb_index)
    # writer.add_scalar('Metrics/MCC', score["mcc"], tb_index)

In [59]:
# 初始化TensorBoard SummaryWriter
writer = SummaryWriter(data_output_dir_path / f'runs/multi-model/{mode_name}')

from sklearn.model_selection import KFold

# TODO 迭代次数
num_epochs = 5
k_folds = 5  # 设置K折交叉验证的折数
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

fold_size = len(all_dataset) // k_folds


In [60]:
sum_tensor = torch.zeros((1, 7))
for fold, (train_indices, val_indices) in enumerate(kf.split(all_dataset)):
    # print(f"Fold {fold + 1}/{k_folds}")

    # 创建fold特定的train和val loader
    train_fold_data = [all_dataset[i] for i in train_indices]
    val_fold_data = [all_dataset[i] for i in val_indices]

    train_fold_loader = torch.utils.data.DataLoader(train_fold_data, batch_size=train_loader.batch_size, shuffle=True)
    val_fold_loader = torch.utils.data.DataLoader(val_fold_data, batch_size=val_loader.batch_size, shuffle=False)
    

    for epoch in range(num_epochs):
        correct = 0
        total = 0
        # 初始化变量来存储所有预测值和标签
        all_labels = []
        all_preds_prob = []

        model.train()
        # 导入 batch 数据
        for batch in train_fold_loader:
            gene_tensor = batch["gene_tensor"].to(device)
            cnv_tensor = batch["cnv_tensor"].to(device)
            report_tensor = batch["report_tensor"].to(device)
            wsi_tensor = batch["wsi_tensor"].to(device)
            label_tensor = torch.squeeze(batch["label_tensor"]).to(torch.float32).to(device).view(-1, 1)
            label_tensor = label_tensor.squeeze()
            label_tensor = label_tensor.long()
            
            outputs = model(gene_tensor, cnv_tensor, report_tensor, wsi_tensor)

            optimizer.zero_grad()
            loss = criterion(outputs, label_tensor)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)  # 获取预测的类别
            total += label_tensor.size(0)
            correct += (preds == label_tensor).sum().item()

        # print("training loss: ", loss.item())
        score = get_model_score(outputs, label_tensor.view(-1))
        score_tensor = show_score(score, can_print=True, tb_index=epoch, fold_idx=fold, epoch_idx=epoch)
        sum_tensor += score_tensor
        # writer.add_scalar('training/training_loss', loss.item(), epoch)

        # 计算准确率
        accuracy = correct / total
        # print(f"training Accuracy: ({accuracy})")
        # writer.add_scalar('training/train_accuracy', accuracy, epoch)

        # # 记录训练损失和准确率到同一个图表的不同系列
        # writer.add_scalars('loss', {
        #     'train': loss.item(),
        # }, epoch + fold * num_epochs)  # 调整epoch计数

        # writer.add_scalars('accuracy', {
        #     'train': accuracy,
        # }, epoch + fold * num_epochs)  # 调整epoch计数

        correct = 0
        total = 0
        # 初始化变量来存储所有预测值和标签
        all_labels = []
        all_preds_prob = []

        # 在每个epoch结束时进行验证
        model.eval()
        with torch.no_grad():
            for batch in val_fold_loader:
                gene_tensor = batch["gene_tensor"].to(device)
                cnv_tensor = batch["cnv_tensor"].to(device)
                report_tensor = batch["report_tensor"].to(device)
                wsi_tensor = batch["wsi_tensor"].to(device)
                label_tensor = torch.squeeze(batch["label_tensor"]).to(torch.float32).to(device).view(-1, 1)
                label_tensor = label_tensor.squeeze()
                label_tensor = label_tensor.long()

                outputs = model(gene_tensor, cnv_tensor, report_tensor, wsi_tensor)

                loss = criterion(outputs, label_tensor)

                # outputs = model(gene_tensor)
                _, preds = torch.max(outputs, 1)  # 获取预测的类别
                total += label_tensor.size(0)
                correct += (preds == label_tensor).sum().item()
                
                # 假设outputs是模型的输出，形状为[24, 2]
                # 我们只关心正类的概率，所以取第二列（索引为1）
                all_preds_prob.extend(torch.sigmoid(outputs)[:, 1].cpu().numpy())
                all_labels.extend(label_tensor.cpu().numpy())

        # writer.add_scalars('loss', {
        #     'valid': loss.item(),
        # }, epoch + fold * num_epochs)  # 调整epoch计数

        # 计算准确率
        accuracy = correct / total
        # print(f"Validation Accuracy: ({accuracy})")
        # writer.add_scalar('validation/validation_accuracy', accuracy, epoch)

        # 计算AUC
        auc = roc_auc_score(all_labels, all_preds_prob)
        # print(f'AUC: {auc}')
        # writer.add_scalar('validation/validation_auc', auc, epoch)

        # writer.add_scalars('accuracy', {
        #     'valid': accuracy,
        # }, epoch + fold * num_epochs)  # 调整epoch计数

        # writer.add_scalars('auc', {
        #     'valid': auc,
        # }, epoch + fold * num_epochs)  # 调整epoch计数

writer.close()

mean_tensor = sum_tensor / (k_folds * num_epochs)
mean_list = mean_tensor.detach().cpu().flatten().tolist()


print("最终平均结果:")
headers = ["      ","Auc", "Acc", "F1-score", "Pre", "Sn", "Sp", "Mcc"]
for h in headers:
    print(f"{h:<10}", end=" ")
print()
# Print data
model_name = "Genomic"
print(f"{model_name:<10}", end=" ")
for s in mean_list:
    print(f"{s:<10.4f}", end=' ')

1-1 0.4423 0.5600 0.6452 0.5556 0.7692 0.3333 0.1141
1-2 0.5694 0.5200 0.6000 0.6429 0.5625 0.4444 0.0067
1-3 0.6250 0.4800 0.3158 0.3000 0.3333 0.5625 -0.1021
1-4 0.6538 0.5200 0.6000 0.5294 0.6923 0.3333 0.0275
1-5 0.5147 0.6400 0.6897 0.8333 0.5882 0.7500 0.3158
2-1 0.7051 0.7200 0.7407 0.7143 0.7692 0.6667 0.4387
2-2 0.7133 0.6000 0.6154 0.5000 0.8000 0.4667 0.2722
2-3 0.8333 0.8000 0.8148 0.7857 0.8462 0.7500 0.6000
2-4 0.8247 0.7200 0.6667 0.7000 0.6364 0.7857 0.4277
2-5 0.8397 0.9200 0.9231 0.9231 0.9231 0.9167 0.8397
3-1 0.7857 0.6923 0.7143 0.6250 0.8333 0.5714 0.4148
3-2 0.6545 0.6154 0.6154 0.7273 0.5333 0.7273 0.2606
3-3 0.9812 0.9231 0.8889 1.0000 0.8000 1.0000 0.8433
3-4 0.8938 0.8462 0.7778 0.8750 0.7000 0.9375 0.6720
3-5 0.7857 0.8846 0.8966 0.8667 0.9286 0.8333 0.7688
4-1 0.9394 0.9615 0.9565 0.9167 1.0000 0.9333 0.9250
4-2 0.9515 0.8846 0.8966 0.9286 0.8667 0.9091 0.7688
4-3 0.8681 0.8077 0.7059 0.6667 0.7500 0.8333 0.5659
4-4 0.9822 1.0000 1.0000 1.0000 1.0000 1.0000